### Utility Functions

In [ ]:
from wave_breaking import (
    MASK_THRESHOLD,
    MASK_TRIM_END,
    MASK_TRIM_START,
    SMOOTH_P_T,
    SMOOTH_P_X,
    build_case_dataset,
    check_periodicity_assumption,
)


### Imports

In [ ]:
import os
from datetime import datetime

import numpy as np
from scipy.io import savemat
from scipy.signal import hilbert
from pymatreader import read_mat
import matplotlib.pyplot as plt

### Reproducible dataset extraction and grouped validation


In [ ]:
DATA_DIR = r"data"
OUTPUT_DIR = r"final_dataset"
RUN_PERIODICITY_CHECK = True
print(
    f"Shared settings: p_x={SMOOTH_P_X}, p_t={SMOOTH_P_T}; "
    f"mask threshold={MASK_THRESHOLD}, trim={MASK_TRIM_START}/{MASK_TRIM_END} columns"
)

file_names = sorted(
    name for name in os.listdir(DATA_DIR)
    if name.lower().endswith(".mat")
    and os.path.isfile(os.path.join(DATA_DIR, name))
)
if not file_names:
    raise FileNotFoundError(f"No .mat source files found in {DATA_DIR!r}")

feature_chunks = []
target_full_chunks = []
target_residual_chunks = []
source_id_chunks = []

for source_id, name in enumerate(file_names):
    fpath = os.path.join(DATA_DIR, name)
    data = read_mat(fpath)
    sfr3 = data["sfr3"]
    time_vec = np.asarray(data["time_vec"], dtype=float).ravel()
    x = np.asarray(data["x1"], dtype=float).ravel()
    time_len_min = int(data["time_len_min"])
    time_vec = time_vec[:time_len_min]

    for key in list(sfr3):
        sfr3[key] = np.asarray(sfr3[key])[:, :time_len_min]

    if RUN_PERIODICITY_CHECK:
        check_periodicity_assumption(sfr3["eta"], label=f"{name}: eta")
        check_periodicity_assumption(sfr3["ux"], label=f"{name}: ux")

    wave_period = float(name.split("_")[3].replace("d", "."))
    case = build_case_dataset(sfr3, time_vec, x, wave_period)
    n_points = case["features"].shape[0]
    feature_chunks.append(case["features"])
    target_full_chunks.append(case["target_full"])
    target_residual_chunks.append(case["target_residual"])
    source_id_chunks.append(np.full(n_points, source_id, dtype=np.int16))
    print(f"Processed {name}: added {n_points} points")

if not feature_chunks or sum(len(chunk) for chunk in feature_chunks) == 0:
    raise ValueError("No points passed the selection mask; check source data and mask settings")

u_in = np.concatenate(feature_chunks, axis=0)
tar_eta_full = np.concatenate(target_full_chunks)
tar_eta_residual = np.concatenate(target_residual_chunks)
source_id = np.concatenate(source_id_chunks)
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(
    OUTPUT_DIR, f"db_database_full_discover_{datetime.now():%d%m%y_%H%M%S}.mat"
)
savemat(
    out_path,
    {
        "u_in": u_in,
        "tar_eta_full": tar_eta_full,
        "tar_eta_residual": tar_eta_residual,
        "source_id": source_id,
        "source_file_names": np.asarray(file_names, dtype=object),
    },
)
print(f"Saved {len(u_in):,} rows from {len(file_names)} files to {out_path}")


### Training

In [ ]:
f = read_mat(out_path)

X_full = np.asarray(f["u_in"], dtype=float)
y_res = np.asarray(f["tar_eta_residual"], dtype=float).ravel()
y_full = np.asarray(f["tar_eta_full"], dtype=float).ravel()
if "source_id" not in f:
    raise ValueError("Dataset has no source_id metadata; rerun the extraction cell before validation")
source_id = np.asarray(f["source_id"], dtype=int).ravel()
if not (len(X_full) == len(y_res) == len(y_full) == len(source_id)):
    raise ValueError("Feature, target, and source_id row counts do not match")

k, omega = X_full[:, 13], X_full[:, 14]
X_tilde = np.zeros_like(X_full)
X_tilde[:, 0] = X_full[:, 0]
X_tilde[:, 1] = X_full[:, 1] * (k / omega)
X_tilde[:, 2] = X_full[:, 2] * (k / omega)
X_tilde[:, 3] = X_full[:, 3] / omega
X_tilde[:, 5] = X_full[:, 5] / omega
X_tilde[:, 7] = X_full[:, 7] / k
X_tilde[:, 8] = X_full[:, 8] * k
X_tilde[:, 9] = X_full[:, 9] * k
X_tilde[:, 10] = X_full[:, 10] * k
X_tilde[:, 11] = X_full[:, 11] * k
X_tilde[:, 12] = X_full[:, 12] * k

y_res_tilde = y_res * (k / omega**2)
y_full_tilde = y_full * (k / omega**2)


In [ ]:
meow = X_tilde[:,7]
np.min(meow)
np.sum(np.abs(meow) > 50) / len(meow)

In [ ]:
if not (np.isfinite(X_tilde).all() and np.isfinite(y_res_tilde).all()):
    raise ValueError("Transformed features or target contain NaN or infinite values")
print(
    f"Prepared {len(X_tilde):,} rows, {X_tilde.shape[1]} stored features, "
    f"and {unique_sources.size} source-file groups."
)


### Validation and feature dependencies
Space-time rows from one wave file are correlated. Keep complete source files out of each validation fold; row-wise random splits can overstate generalization. Compare models and curvature variants on training-file folds, then use the untouched file holdout for permutation importance and final scores.


In [ ]:
# Keep the established feature set, and compare it with the same set plus curvature.
feature_cols = [0, 1, 2, 3, 5, 8, 9, 10, 11, 12]
feature_names_by_col = {
    0: "deta_dx",
    1: "uy_tilde",
    2: "ux_tilde",
    3: "dudx_tilde",
    5: "duydx_tilde",
    7: "curvature_tilde",
    8: "hilbert_t_img_tilde",
    9: "hilbert_x_img_tilde",
    10: "hilbert_t_abs_tilde",
    11: "hilbert_x_abs_tilde",
    12: "eta_tilde",
}
feature_names = [feature_names_by_col[i] for i in feature_cols]
X_pysr = X_tilde[:, feature_cols]

from sklearn.model_selection import GroupShuffleSplit

unique_sources = np.unique(source_id)
if unique_sources.size < 3:
    raise ValueError("At least three source files with selected points are needed for grouped validation")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(
    splitter.split(X_pysr, y_res_tilde, groups=source_id)
)
train_groups = np.unique(source_id[train_idx])
test_groups = np.unique(source_id[test_idx])
if np.intersect1d(train_groups, test_groups).size:
    raise RuntimeError("Source-file leakage detected between train and test groups")
X_train, y_train = X_pysr[train_idx], y_res_tilde[train_idx]
X_test, y_test = X_pysr[test_idx], y_res_tilde[test_idx]
print(
    f"Grouped split: {len(train_idx):,} train / {len(test_idx):,} test rows; "
    f"{len(train_groups)} train / {len(test_groups)} held-out source files."
)


In [ ]:
from pysr import PySRRegressor
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(42)
pysr_train_idx = rng.choice(train_idx, size=min(10_000, len(train_idx)), replace=False)
model = PySRRegressor(
    niterations=100,
    populations=30,
    population_size=40,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square", "cube", "abs", "sin", "cos", "tanh", "exp"],
    parsimony=0.005,
    maxsize=20,
    parallelism="multithreading",
    random_state=42,
)
model.fit(
    X_pysr[pysr_train_idx],
    y_res_tilde[pysr_train_idx],
    variable_names=feature_names,
)
y_pred_test = model.predict(X_test)
print("Best equation:", model.get_best()["equation"])
print(f"Grouped held-out R^2: {r2_score(y_test, y_pred_test):.4f}")
print(f"Grouped held-out RMSE: {mean_squared_error(y_test, y_pred_test) ** 0.5:.6f}")


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_templates = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=100, max_depth=3, min_samples_leaf=5, random_state=42
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=100, max_features=0.8, min_samples_leaf=3,
        n_jobs=-1, random_state=42
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=150, max_leaf_nodes=31, l2_regularization=1.0, random_state=42
    ),
}
try:
    from xgboost import XGBRegressor
except ImportError:
    print(
        "XGBoost is not installed; the previous notebook's 0.89 model was "
        "sklearn GradientBoostingRegressor, not XGBoost."
    )
else:
    model_templates["XGBoost"] = XGBRegressor(
        n_estimators=150, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        objective="reg:squarederror", n_jobs=-1, random_state=42,
    )

feature_sets = {
    "without curvature": feature_cols,
    "with curvature": feature_cols + [7],
}
n_splits = min(3, len(train_groups))
group_cv = GroupKFold(n_splits=n_splits)
benchmark_rows = []

for feature_set_name, cols in feature_sets.items():
    X_candidate = X_tilde[train_idx][:, cols]
    y_candidate = y_res_tilde[train_idx]
    groups_candidate = source_id[train_idx]
    names = [feature_names_by_col[i] for i in cols]
    for model_name, template in model_templates.items():
        fold_scores = []
        for fold_train, fold_test in group_cv.split(
            X_candidate, y_candidate, groups=groups_candidate
        ):
            fitted = clone(template)
            fitted.fit(X_candidate[fold_train], y_candidate[fold_train])
            prediction = fitted.predict(X_candidate[fold_test])
            fold_scores.append(r2_score(y_candidate[fold_test], prediction))
        benchmark_rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "mean_grouped_r2": float(np.mean(fold_scores)),
            "std_grouped_r2": float(np.std(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0,
            "fold_r2": fold_scores,
            "columns": cols,
            "feature_names": names,
        })

benchmark_rows.sort(key=lambda row: row["mean_grouped_r2"], reverse=True)
for row in benchmark_rows:
    print(
        f"{row['feature_set']:<18} {row['model']:<22} "
        f"grouped R^2 = {row['mean_grouped_r2']:.4f} +/- {row['std_grouped_r2']:.4f} "
        f"{row['fold_r2']}"
    )
best_row = benchmark_rows[0]
best_model_name = best_row["model"]
best_cols = best_row["columns"]
best_feature_names = best_row["feature_names"]
print(
    f"Best grouped-CV model: {best_model_name} ({best_row['feature_set']}); "
    "folds hold out entire source files."
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

best_model = clone(model_templates[best_model_name])
best_model.fit(X_tilde[train_idx][:, best_cols], y_res_tilde[train_idx])

rng = np.random.default_rng(43)
importance_idx = rng.choice(test_idx, size=min(20_000, len(test_idx)), replace=False)
permutation = permutation_importance(
    best_model,
    X_tilde[importance_idx][:, best_cols],
    y_res_tilde[importance_idx],
    scoring="r2",
    n_repeats=3,
    random_state=42,
    n_jobs=1,
)
importance_order = np.argsort(permutation.importances_mean)[::-1]
print("Held-out grouped permutation importance (mean +/- std):")
for i in importance_order:
    print(
        f"  {best_feature_names[i]:<24} "
        f"{permutation.importances_mean[i]:.4f} +/- {permutation.importances_std[i]:.4f}"
    )

sample_idx = rng.choice(train_idx, size=min(50_000, len(train_idx)), replace=False)
dependency_frame = pd.DataFrame(
    X_tilde[sample_idx][:, best_cols], columns=best_feature_names
)
dependency_frame["target"] = y_res_tilde[sample_idx]
correlations = dependency_frame.corr(method="spearman")
print("Spearman correlations with target (training-source sample):")
print(correlations["target"].drop("target").sort_values(key=np.abs, ascending=False))

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(correlations, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlations.columns)), correlations.columns, rotation=75, ha="right")
ax.set_yticks(range(len(correlations.index)), correlations.index)
fig.colorbar(image, ax=ax, label="Spearman correlation")
ax.set_title("Feature and target dependencies (training files only)")
fig.tight_layout()
plt.show()

top_features = importance_order[:min(4, len(importance_order))]
pd_idx = rng.choice(train_idx, size=min(10_000, len(train_idx)), replace=False)
fig, axes = plt.subplots(1, len(top_features), figsize=(5 * len(top_features), 4), squeeze=False)
PartialDependenceDisplay.from_estimator(
    best_model,
    X_tilde[pd_idx][:, best_cols],
    features=list(top_features),
    feature_names=best_feature_names,
    kind="average",
    ax=axes[0],
    n_jobs=1,
)
fig.suptitle("Grouped-training partial dependence for top held-out features")
fig.tight_layout()
plt.show()


### Symbolic distillation of the validated regression model
Cross-fitted teacher predictions on training files are the symbolic-regression target. Report symbolic performance against both the teacher and the physical residual on held-out files.


In [ ]:
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from pysr import PySRRegressor

# Cross-fitted teacher predictions prevent PySR from learning the teacher's
# in-sample training residuals. All folds are grouped by source file.
X_teacher = X_tilde[:, best_cols]
teacher_model_name = "XGBoost" if "XGBoost" in model_templates else best_model_name
if teacher_model_name != "XGBoost":
    print(f"Using {teacher_model_name} as the teacher because XGBoost is unavailable")
teacher_template = model_templates[teacher_model_name]
train_source_ids = source_id[train_idx]
teacher_splits = min(3, np.unique(train_source_ids).size)
if teacher_splits < 2:
    raise ValueError("At least two training source files are needed for teacher cross-fitting")
teacher_oof = cross_val_predict(
    clone(teacher_template),
    X_teacher[train_idx],
    y_res_tilde[train_idx],
    groups=train_source_ids,
    cv=GroupKFold(n_splits=teacher_splits),
    n_jobs=1,
)
teacher = clone(teacher_template)
teacher.fit(X_teacher[train_idx], y_res_tilde[train_idx])
teacher_test = teacher.predict(X_teacher[test_idx])
print(f"Teacher grouped held-out R^2 vs physical target: {r2_score(y_test, teacher_test):.4f}")

augmented_features = [X_teacher]
augmented_names = list(best_feature_names)
if "uy_tilde" in best_feature_names:
    uy_index = best_feature_names.index("uy_tilde")
    uy = X_teacher[:, uy_index]
    augmented_features.extend((np.sign(uy)[:, None], np.tanh(50 * uy)[:, None]))
    augmented_names.extend(("sign_uy", "tanh50_uy"))
if "uy_tilde" in best_feature_names and "dudx_tilde" in best_feature_names:
    dudx_index = best_feature_names.index("dudx_tilde")
    interaction = X_teacher[:, uy_index] * X_teacher[:, dudx_index]
    augmented_features.append(interaction[:, None])
    augmented_names.append("uy_times_dudx")
X_teacher_aug = np.column_stack(augmented_features)

rng = np.random.default_rng(42)
train_positions = rng.choice(
    len(train_idx), size=min(10_000, len(train_idx)), replace=False
)
model_aug = PySRRegressor(
    niterations=100,
    populations=30,
    population_size=40,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square", "cube", "abs", "sin", "cos", "tanh", "exp"],
    parsimony=0.001,
    maxsize=20,
    random_state=42,
)
model_aug.fit(
    X_teacher_aug[train_idx][train_positions],
    teacher_oof[train_positions],
    variable_names=augmented_names,
)
symbolic_test = model_aug.predict(X_teacher_aug[test_idx])
print("Distilled equation:", model_aug.get_best()["equation"])
print(f"Symbolic R^2 vs teacher predictions: {r2_score(teacher_test, symbolic_test):.4f}")
print(f"Symbolic R^2 vs physical target: {r2_score(y_test, symbolic_test):.4f}")
print(
    f"Symbolic RMSE vs physical target: "
    f"{mean_squared_error(y_test, symbolic_test) ** 0.5:.6f}"
)

import matplotlib.pyplot as plt
plot_idx = rng.choice(len(test_idx), size=min(25_000, len(test_idx)), replace=False)
plot_pairs = [
    (y_test[plot_idx], teacher_test[plot_idx], "Physical target", "Teacher prediction"),
    (teacher_test[plot_idx], symbolic_test[plot_idx], "Teacher prediction", "PySR prediction"),
    (y_test[plot_idx], symbolic_test[plot_idx], "Physical target", "PySR prediction"),
]
fig, axes = plt.subplots(1, len(plot_pairs), figsize=(15, 4.5))
for ax, (actual, predicted, x_label, y_label) in zip(axes, plot_pairs):
    ax.scatter(actual, predicted, s=3, alpha=0.2)
    lower = min(np.min(actual), np.min(predicted))
    upper = max(np.max(actual), np.max(predicted))
    ax.plot([lower, upper], [lower, upper], "r--", linewidth=1)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.grid(True, alpha=0.25)
fig.suptitle("Grouped held-out regression and symbolic-distillation parity")
fig.tight_layout()
plt.show()
